# Shorts render speed test on Kaggle

Clones the public repo, installs Node 20 + dependencies, builds `script_rocket_fire`, and **times every step** including the Remotion render.

**Before running:** set **Internet: On** in the right sidebar (Settings → Internet).

GPU is optional — `--gl=swangle` is software rendering, so a GPU won't change the result. To try the GPU path, change `--gl=swangle` to `--gl=angle` in the render cell.

Edit `SCRIPT` and `CONCURRENCIES` in the setup cell. Set `CONCURRENCIES = [2, 4, 8]` to compare render speeds.

Everything is saved under `/kaggle/working/shorts-studio/out/`.

In [ ]:
import os, subprocess, time, shutil
from IPython.display import display, Video, FileLink

SCRIPT = 'script_rocket_fire'   # any tools/script_*.json
CONCURRENCIES = [4]             # try [2, 4, 8] to compare speeds
REPO = '/kaggle/working/shorts-studio'
NODE_TAR = 'node-v20.19.0-linux-x64'
NODE_DIR = f'/kaggle/working/{NODE_TAR}/bin'
SUDO = '' if os.geteuid() == 0 else 'sudo '

os.environ['PATH'] = NODE_DIR + ':' + os.environ['PATH']
TIMINGS = {}

def timed(name, cmd, cwd=None):
    print(f'\n=== {name} ===')
    print(f'$ {cmd}')
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, cwd=cwd, executable='/bin/bash')
    dt = time.time() - t0
    TIMINGS[name] = TIMINGS.get(name, 0) + dt
    print(f'--- {name}: {dt:.1f}s (exit {r.returncode}) ---')
    if r.returncode != 0:
        raise RuntimeError(f'{name} failed with exit {r.returncode}')
    return dt

In [ ]:
if not os.path.exists(f'{NODE_DIR}/node'):
    timed('install Node 20', f'cd /kaggle/working && curl -fsSLO https://nodejs.org/dist/v20.19.0/{NODE_TAR}.tar.xz && tar -xf {NODE_TAR}.tar.xz')
os.environ['PATH'] = NODE_DIR + ':' + os.environ['PATH']
print('node:', subprocess.check_output(['node', '-v'], text=True).strip())
print('npm :', subprocess.check_output(['npm', '-v'], text=True).strip())

In [ ]:
if not shutil.which('ffmpeg'):
    timed('install ffmpeg', f'{SUDO}apt-get update -qq && {SUDO}apt-get install -y -qq ffmpeg')
timed('pip edge-tts', 'pip install -q edge-tts')
print('ffmpeg:', subprocess.check_output(['ffmpeg', '-version'], text=True).splitlines()[0])

In [ ]:
if not os.path.isdir(REPO):
    timed('clone repo', f'git clone -q https://github.com/N-A-I-T-I-K/shorts-studio.git {REPO}')
os.chdir(REPO)
print('repo ready at', os.getcwd())

In [ ]:
timed('npm ci', 'npm ci', cwd=REPO)

In [ ]:
timed('fetch photos', f'python tools/fetch_assets.py tools/{SCRIPT}.json', cwd=REPO)

In [ ]:
timed('build story', f'python tools/build_story.py tools/{SCRIPT}.json', cwd=REPO)
import json
d = json.load(open('public/story.json', encoding='utf8'))
print('duration:', d['totalSec'], 's | words:', sum(len(s['words']) for s in d['scenes']), '| photos:', sum(len(s['photos']) for s in d['scenes']))

In [ ]:
for c in CONCURRENCIES:
    out = f'out/{SCRIPT}_c{c}.mp4'
    timed(f'render c={c}', f'npx remotion render StoryFlow {out} --gl=swangle --concurrency={c}', cwd=REPO)

In [ ]:
import pandas as pd
df = pd.DataFrame([{'step': k, 'seconds': round(v, 1)} for k, v in TIMINGS.items()])
display(df)
print('TOTAL:', round(sum(TIMINGS.values()), 1), 's')

best = min(CONCURRENCIES, key=lambda c: TIMINGS.get(f'render c={c}', 1e9))
video = f'{REPO}/out/{SCRIPT}_c{best}.mp4'
print('fastest render: concurrency', best, '->', video)
display(Video(video, width=360))
display(FileLink(video))